# Deep Neural Network for MNIST Classification

The dataset provided here is called MNIST and refers to handwritten digit recognition. You can find more about it on Yann LeCun's website. He is one of the pioneers of what we've been talking about and of more complex approaches that are widely used today, such as Convolutional Neural Networks (CNNs).

The dataset provides 70,000 images (28x28 pixels) of handwritten digits (1 digit per image).

The goal is to write an algorithm that detects which digit is written. Since there are only 10 digits (0, 1, 2, 3, 4, 5, 6, 7, 8, 9), this is a classification problem with 10 classes.

Our goal would be to build a neural network with 2 hidden layers.

## Import the relevant packages

In [3]:
# TensorFLow includes a data provider for MNIST.
# It comes with the tensorflow-datasets module
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

## Data

Load and preprocess our data.

In [4]:
# tfds.load actually loads a dataset
# The data is stored on the local file system of the Colab runtime environment (virtual machine)
# The name of the dataset is the only mandatory argument
# with_info=True provides a tuple containing information about version, features, number of samples
# as_supervised=True will load the dataset in a 2-tuple structure (input, target)
# as_supervised=False, would return a dictionary
# we will use this information a bit below and we will store it in mnist_info

mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)

# extract the training and testing dataset
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

# MNIST has training and testing datasets, but no validation sets
# split by defining the number of validation samples as n% of the train samples
# make use of mnist_info
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples

# Set to an integer, float may cause an error
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

# Store the number of test samples in a dedicated variable
num_test_samples = mnist_info.splits['test'].num_examples

# Prefer an integer
num_test_samples = tf.cast(num_test_samples, tf.int64)

# Scale the data to make the result numerically stable
def scale(image, label):
    image = tf.cast(image, tf.float32)
    # make sure the value is a float
    # possible values for the inputs are 0 to 255
    # divide each element by 255
    # all elements will be between 0 and 1
    image /= 255.
    return image, label

# The method .map() allows us to apply a custom transformation
scaled_train_and_validation_data = mnist_train.map(scale)

# Scale the test data
test_data = mnist_test.map(scale)

# Shuffle the data
# the BUFFER_SIZE parameter is here for cases with enormous datasets
# we can't shuffle the whole dataset in one go because we can't fit it all in memory
# instead TF only stores BUFFER_SIZE samples in memory at a time and shuffles them
# if BUFFER_SIZE=1 => no shuffling will actually happen
# there is a shuffle method available and we just need to specify the buffer size
BUFFER_SIZE = 10000
shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

# extract the train and validation data set
# validation data would be equal to 10% of the training set
# we use the .take() method to take that many samples
validation_data = shuffled_train_and_validation_data.take(num_validation_samples)

# the train_data is everything else
train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

# determine the batch size
# batch the train data
# it is helpful to iterate over the different batches
BATCH_SIZE = 100
train_data = train_data.batch(BATCH_SIZE)
validation_data = validation_data.batch(num_validation_samples)

# batch the test data
test_data = test_data.batch(num_test_samples)

# takes next batch, it is the only batch
# because as_supervised=True we receive a 2-tuple structure
validation_inputs, validation_targets = next(iter(validation_data))

## Model

### Outline the model

In [8]:
input_size = 784
output_size = 10

# use same hidden layer size for both hidden layers
hidden_layer_size = 50

# define the model
model = tf.keras.Sequential([

    # the first layer (the input layer)
    # each observation is 28x28x1 pixels, a tensor of rank 3
    # Skip CNN, flatten the images, 28x28x1 tensor, (784,) vector
    # approach is call feed forward neural network
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),

    # important arguments are the hidden_layer_size and the activation function
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),

    # final layer = output layeer, activate with softmax
    tf.keras.layers.Dense(output_size, activation='softmax') # output layer
])

### Choose the optimizer and the loss function

In [9]:
# Define optimizer, loss function, metrics at each iteration
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

### Training
That's where we train the model we have built.

In [11]:
# Determine the maximum number of epochs
NUM_EPOCHS = 5

# Fit the model, specify the training data, number of epochs, validation data
# Try different hyperparams later
model.fit(train_data, epochs=NUM_EPOCHS, validation_data=(validation_inputs, validation_targets), validation_steps=10, verbose =2)

Epoch 1/5
540/540 - 4s - 7ms/step - accuracy: 0.9764 - loss: 0.0777 - val_accuracy: 0.9742 - val_loss: 0.0875
Epoch 2/5
540/540 - 6s - 11ms/step - accuracy: 0.9787 - loss: 0.0697 - val_accuracy: 0.9727 - val_loss: 0.0880
Epoch 3/5
540/540 - 9s - 16ms/step - accuracy: 0.9806 - loss: 0.0630 - val_accuracy: 0.9798 - val_loss: 0.0693
Epoch 4/5
540/540 - 6s - 10ms/step - accuracy: 0.9827 - loss: 0.0558 - val_accuracy: 0.9802 - val_loss: 0.0725
Epoch 5/5
540/540 - 3s - 6ms/step - accuracy: 0.9848 - loss: 0.0502 - val_accuracy: 0.9807 - val_loss: 0.0666


## Test the model

After training on the training data and validating on the validation data, test the final prediction power of the model by running it on the test dataset that the algorithm has NEVER seen before. The test is the absolute final instance. You should not test before you are completely done with adjusting your model.

In [12]:
test_loss, test_accuracy = model.evaluate(test_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.9726 - loss: 0.0915


In [13]:
# Convert to a better format
print('Test loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

Test loss: 0.09. Test accuracy: 97.26%


Using the initial model and hyperparameters given in this notebook, the final test accuracy should be roughly around 97%. Each time the code is rerun, we get a different accuracy as the batches are shuffled and the weights are initialized in a different way.

Finally, we have intentionally reached a suboptimal solution:
- Try to optimize the NN with different hyperparameters (width, depth, etc.)
- Search for extensions of the current code with Convolutional Neural Networks (CNN)